In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv("../datos/predictive_maintenance.csv")

print("Dimensiones iniciales:", df.shape)
display(df.head())
df.info()
display(df.describe())
display(df.isnull().sum())

df_limpio = df.copy()
df_limpio.columns = df_limpio.columns.str.strip()
duplicados = df_limpio.duplicated().sum()
print("Duplicados exactos encontrados:", duplicados)
df_limpio = df_limpio.drop_duplicates()
print("Valores faltantes por columna:")
display(df_limpio.isnull().sum())
columnas_numericas = df_limpio.select_dtypes(include=[np.number]).columns

for col in columnas_numericas:
    df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())

columnas_categoricas = df_limpio.select_dtypes(include=["object"]).columns

for col in columnas_categoricas:
    df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode()[0])

df_limpio["Type"] = df_limpio["Type"].astype(str).str.strip().str.upper()
df_limpio["Failure Type"] = df_limpio["Failure Type"].astype(str).str.strip()

print("Categorías en Type:")
display(df_limpio["Type"].unique())

print("Categorías en Failure Type:")
display(df_limpio["Failure Type"].unique())

df_limpio = df_limpio[df_limpio["Air temperature [K]"] > 0]
df_limpio = df_limpio[df_limpio["Process temperature [K]"] > 0]
df_limpio = df_limpio[df_limpio["Rotational speed [rpm]"] >= 0]
df_limpio = df_limpio[df_limpio["Torque [Nm]"] >= 0]
df_limpio = df_limpio[df_limpio["Tool wear [min]"] >= 0]

df_limpio = df_limpio[df_limpio["Target"].isin([0, 1])]

df_limpio = df_limpio.drop_duplicates(
    subset=["Product ID", "Type", "Air temperature [K]", "Process temperature [K]"]
)

def detectar_outliers_iqr(dataframe, columna):
    Q1 = dataframe[columna].quantile(0.25)
    Q3 = dataframe[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = dataframe[
        (dataframe[columna] < limite_inferior) |
        (dataframe[columna] > limite_superior)
    ]

    print(f"{columna}: {len(outliers)} outliers detectados")

    return limite_inferior, limite_superior

variables_outliers = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

for col in variables_outliers:
    detectar_outliers_iqr(df_limpio, col)

print("Dimensiones finales:", df_limpio.shape)

os.makedirs("../resultados", exist_ok=True)

df_limpio.to_csv(
    "../resultados/dataset_limpio.csv",
    index=False
)

print("Dataset limpio guardado correctamente.")

display(df_limpio.info())
display(df_limpio.describe())

missing = df_limpio.isnull().sum()
display(missing)

plt.figure(figsize=(10, 5))
sns.heatmap(df_limpio.isnull(), cbar=False)
plt.title("Mapa de valores faltantes")
plt.show()

media = df_limpio.mean(numeric_only=True)
mediana = df_limpio.median(numeric_only=True)
moda = df_limpio.mode().iloc[0]

print("Media:")
display(media)

print("Mediana:")
display(mediana)

print("Moda:")
display(moda)

df_limpio.hist(figsize=(15, 10))
plt.suptitle("Histogramas de variables numéricas")
plt.tight_layout()
plt.show()

for col in variables_outliers:
    plt.figure(figsize=(8, 5))
    sns.histplot(df_limpio[col], kde=True)
    plt.title(f"Distribución de {col}")
    plt.show()

for col in variables_outliers:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df_limpio[col])
    plt.title(f"Boxplot de {col}")
    plt.show()

corr = df_limpio.corr(numeric_only=True)

display(corr)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Mapa de correlación de variables numéricas")
plt.tight_layout()
plt.savefig("../resultados/mapa_correlacion.png")
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(x="Target", data=df_limpio)
plt.title("Distribución de fallos de máquina")
plt.xlabel("Target: 0 = Sin fallo, 1 = Con fallo")
plt.ylabel("Cantidad de registros")
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(x="Type", hue="Target", data=df_limpio)
plt.title("Fallos por tipo de máquina")
plt.xlabel("Tipo de máquina")
plt.ylabel("Cantidad")
plt.show()

plt.figure(figsize=(10, 5))
sns.countplot(y="Failure Type", data=df_limpio)
plt.title("Cantidad por tipo de fallo")
plt.xlabel("Cantidad")
plt.ylabel("Tipo de fallo")
plt.show()

for col in variables_outliers:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x="Target", y=col, data=df_limpio)
    plt.title(f"{col} según existencia de fallo")
    plt.xlabel("Target: 0 = Sin fallo, 1 = Con fallo")
    plt.ylabel(col)
    plt.show()

print("RESUMEN FINAL")
print("Registros iniciales:", df.shape[0])
print("Registros finales:", df_limpio.shape[0])
print("Columnas:", df_limpio.shape[1])
print("Valores faltantes finales:", df_limpio.isnull().sum().sum())
print("Duplicados finales:", df_limpio.duplicated().sum())
print("Cantidad de fallos:", df_limpio["Target"].sum())
print("Porcentaje de fallos:", round(df_limpio["Target"].mean() * 100, 2), "%")

